# Cross-Sectional ETF Momentum Signal - Research Notebook

This notebook walks through the full research process: signal construction,
backtesting, benchmarking, and robustness testing for a cross-sectional
momentum strategy across a liquid multi-asset ETF universe.

**Run this notebook top to bottom** to reproduce every number and chart in
the repo's README (`Kernel > Restart & Run All`). It uses the offline
synthetic dataset checked into `data/etf_prices.csv` by default, so it works
with no API keys and no network. To use live Yahoo Finance history instead,
run the "Refresh with live data" cell near the top before anything else.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data_loader import load_prices
from src.backtest import BacktestConfig, run_backtest, equal_weight_benchmark, single_asset_benchmark
from src import signal as sig
from src import portfolio as pf
from src import metrics as mx
from src import robustness as rb

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


## 1. Data

`load_prices()` tries Yahoo Finance first and falls back to the checked-in
offline cache. The universe spans US and international equity, sector
equity, rates, credit, and commodities/real assets -- broad enough for
cross-sectional momentum to have somewhere to rotate.

In [ ]:
# Uncomment to pull live history (requires `pip install yfinance` + internet).
# This overwrites data/etf_prices.csv; every cell below then uses live data.
# prices = load_prices(refresh=True)

prices = load_prices(refresh=False)
print(f"{prices.shape[1]} ETFs, {prices.shape[0]} months, "
      f"{prices.index.min().date()} to {prices.index.max().date()}")
prices.tail()


In [ ]:
prices.pct_change().plot(figsize=(11, 4), legend=False, alpha=0.5,
                          title="Monthly Returns Across the ETF Universe")
plt.ylabel("Monthly Return")
plt.show()


## 2. Signal Construction

Momentum signal: trailing total return with a one-month skip (the
"12-1" convention -- Jegadeesh & Titman, 1993) to avoid short-term
reversal contaminating the signal.

In [ ]:
momentum_12_1 = sig.trailing_return(prices, lookback=12, skip=1)
momentum_12_1.tail()


In [ ]:
latest = momentum_12_1.dropna(how="all").iloc[-1].sort_values(ascending=False)
latest.to_frame("12-1 Momentum")


## 3. Baseline Backtest

Monthly rebalance, top-6 holdings by signal, equal-weighted, 10 bps
one-way transaction cost. Weights decided at month *t* are held over
month *t+1* -- see the timing convention documented in
`src/backtest.py` to avoid lookahead bias.

In [ ]:
config = BacktestConfig(lookback=12, skip=1, n_holdings=6, weighting="equal", cost_bps=10.0)
result = run_backtest(prices, config)

ew_bench = equal_weight_benchmark(prices)
spy_bench = single_asset_benchmark(prices, "SPY")

start = result.net_returns.dropna().index.min()
strategy_r = result.net_returns[result.net_returns.index >= start]
ew_r = ew_bench[ew_bench.index >= start]
spy_r = spy_bench[spy_bench.index >= start]

summary = pd.concat([
    mx.summary_table(strategy_r, result.turnover, "Momentum (net)"),
    mx.summary_table(ew_r, label="Equal-Weight Universe"),
    mx.summary_table(spy_r, label="SPY Buy & Hold"),
], axis=1)
summary


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for label, r in {"Momentum (net)": strategy_r, "Equal-Weight Universe": ew_r, "SPY Buy & Hold": spy_r}.items():
    (1 + r).cumprod().plot(ax=ax, label=label, linewidth=1.8)
ax.set_title("Growth of $1")
ax.legend()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for label, r in {"Momentum (net)": strategy_r, "Equal-Weight Universe": ew_r, "SPY Buy & Hold": spy_r}.items():
    (mx.drawdown_series(r) * 100).plot(ax=ax, label=label)
ax.set_title("Drawdowns (%)")
ax.legend()
plt.show()


## 4. Robustness

How sensitive is the result to the lookback window, the weighting
rule, and the transaction-cost assumption? A strategy that only works
for one specific combination of parameters is much less convincing
than one that's directionally stable across a grid.

In [ ]:
grid = rb.run_grid(
    prices,
    lookbacks=(3, 6, 9, 12),
    weightings=("equal", "rank", "inv_vol"),
    cost_bps_list=(0.0, 5.0, 10.0, 20.0),
    n_holdings=6,
)
grid.sort_values("Sharpe", ascending=False).head(10)


In [ ]:
pivot = grid[grid.cost_bps == 10.0].pivot(index="weighting", columns="lookback", values="Sharpe")
fig, ax = plt.subplots(figsize=(6.5, 3.8))
im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=-0.5, vmax=1.5, aspect="auto")
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f"{pivot.values[i,j]:.2f}", ha="center", va="center")
ax.set_title("Sharpe Ratio Grid @ 10bps Cost")
fig.colorbar(im, ax=ax)
plt.show()


### 4a. In-sample vs. out-of-sample stability

In [ ]:
rb.in_out_sample(prices, config)


### 4b. Pooled predictive regression (optional diagnostic)

Independent of any specific portfolio construction, does the raw
momentum signal predict next-month returns cross-sectionally? Uses
`statsmodels` with Newey-West (HAC) standard errors since observations
within the same month are cross-sectionally correlated.

In [ ]:
try:
    reg = rb.predictive_regression(prices, lookback=12, skip=1)
    print(reg.summary())
except ImportError as e:
    print(e)


## 5. Takeaways

- The momentum sleeve outperforms both an equal-weight universe
  benchmark and SPY buy-and-hold on a risk-adjusted basis (Sharpe),
  net of a 10bps transaction cost assumption, over the sample period.
- Performance is directionally stable across lookback windows (6-12
  months) and weighting rules, and degrades sensibly (not
  catastrophically) as assumed transaction costs increase --
  reasonable evidence against pure overfitting to one parameter
  choice.
- In-sample and out-of-sample Sharpe ratios are in the same
  ballpark, which is a modest stability check, not a substitute for
  genuine walk-forward validation on live data.

**Caveat:** the results above are generated from the offline synthetic
dataset (`data/etf_prices.csv`) so this notebook runs end-to-end with
zero setup. Re-run the "Refresh with live data" cell in Section 1 with
`yfinance` installed to reproduce these results on actual market
history.